# E-Commerce RAG Customer Service Chatbot

**Features:**
- Switch between three retrieval methods (Baseline / Reranker / HyDE)
- Display the retrieved context sources for each response

In [1]:
!pip install -q \
    langchain langchain-community langchain-core \
    langchain-anthropic langchain-huggingface \
    faiss-cpu sentence-transformers gradio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 24.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 35.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 469.4/469.4 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 31.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.0/65.0 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 2.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.0 which is incompatible.


In [2]:
import os
from google.colab import userdata
os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")

from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_anthropic import ChatAnthropic
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser
from sentence_transformers import CrossEncoder

faqs = [
    {"q": "How do I return an item?",
     "a": "You can return items within 15 days of delivery. Go to My Orders, select the order, click Return/Refund. Ensure the item is unused and in original packaging."},
    {"q": "When will I receive my refund?",
     "a": "Refunds are processed within 3-7 business days after the seller confirms the return."},
    {"q": "How do I track my order?",
     "a": "Go to My Orders in the app or website. Click on the order to see real-time tracking updates."},
    {"q": "What payment methods are accepted?",
     "a": "Shopee accepts credit/debit cards, ShopeePay, bank transfer, cash on delivery, and installment payments."},
    {"q": "How do I contact the seller?",
     "a": "Go to the product page and click Chat Now to message the seller directly."},
    {"q": "What is Shopee Guarantee?",
     "a": "Shopee Guarantee holds your payment until you confirm receipt in good condition. This protects buyers from fraud."},
    {"q": "How do I use a voucher?",
     "a": "During checkout, click on the voucher field and enter your code. Discount will be applied automatically."},
    {"q": "My order is delayed, what should I do?",
     "a": "Check tracking first. If no update for 7+ days, contact the seller. If unresolved, file a dispute through Shopee Resolution Centre."},
    {"q": "Can I change my delivery address?",
     "a": "You can change the address before the seller ships. Go to My Orders and click Edit Address if available."},
    {"q": "How do Shopee Coins work?",
     "a": "Shopee Coins are earned from purchases. 100 Coins = 1 unit of local currency discount."},
    {"q": "What is the return window?",
     "a": "The standard return window is 15 days from delivery date. Some sellers offer extended return periods."},
    {"q": "How do I get a refund if item is damaged?",
     "a": "Take photos of the damaged item and go to My Orders > Return/Refund > Damaged Item. Upload photos as evidence."},
    {"q": "Can I get a refund without returning the item?",
     "a": "In some cases yes, if the seller agrees or Shopee decides in your favour during dispute resolution."},
    {"q": "How long does shipping take?",
     "a": "Standard shipping takes 3-7 days. Express shipping takes 1-2 days. International orders may take 7-21 days."},
    {"q": "What if seller doesn't respond?",
     "a": "If the seller does not respond within 3 days, you can escalate to Shopee customer support."},
]

docs = [Document(
    page_content=f"Question: {f['q']}\nAnswer: {f['a']}",
    metadata={"source": "shopee_faq", "question": f['q']}
) for f in faqs]

splitter = RecursiveCharacterTextSplitter(chunk_size=512, chunk_overlap=64)
chunks = splitter.split_documents(docs)

print('loading model...')
embeddings = HuggingFaceEmbeddings(
    model_name='thenlper/gte-small',
    encode_kwargs={'normalize_embeddings': True},
)
vectorstore = FAISS.from_documents(chunks, embeddings)
cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')
llm = ChatAnthropic(model='claude-haiku-4-5-20251001', temperature=0)

RAG_PROMPT = ChatPromptTemplate.from_template("""
You are a helpful Shopee customer service assistant.
Answer the question based ONLY on the following context.
If the answer is not in the context, say "I don't have information about that."

Context:
{context}

Question: {question}
Answer:""")

def format_docs(docs):
    return '\n\n'.join(doc.page_content for doc in docs)


# Reranker
class CrossEncoderReranker:
    def __init__(self, vs, ce, top_n=15, top_k=3):
        self.vs, self.ce, self.top_n, self.top_k = vs, ce, top_n, top_k
    def retrieve(self, query):
        cands = self.vs.similarity_search(query, k=self.top_n)
        scores = self.ce.predict([(query, d.page_content) for d in cands])
        return [d for _, d in sorted(zip(scores, cands), key=lambda x: -x[0])][:self.top_k]

# HyDE
class HyDERetriever:
    def __init__(self, vs, llm, top_k=3):
        self.vs, self.top_k = vs, top_k
        self.gen = (
            ChatPromptTemplate.from_template(
                'Write a short 2-3 sentence answer to this e-commerce question.\n'
                'Question: {question}\nAnswer:')
            | llm | StrOutputParser()
        )
    def retrieve(self, query):
        hypo = self.gen.invoke({'question': query})
        return self.vs.similarity_search(hypo, k=self.top_k)
    def retrieve_with_hypo(self, query):
        hypo = self.gen.invoke({'question': query})
        docs = self.vs.similarity_search(hypo, k=self.top_k)
        return hypo, docs

reranker = CrossEncoderReranker(vectorstore, cross_encoder)
hyde_retriever = HyDERetriever(vectorstore, llm)
baseline_retriever = vectorstore.as_retriever(search_kwargs={'k': 3})

chain_baseline = (
    {'context': baseline_retriever | format_docs, 'question': RunnablePassthrough()}
    | RAG_PROMPT | llm | StrOutputParser()
)
chain_reranked = (
    {'context': RunnableLambda(reranker.retrieve) | format_docs, 'question': RunnablePassthrough()}
    | RAG_PROMPT | llm | StrOutputParser()
)
chain_hyde = (
    {'context': RunnableLambda(hyde_retriever.retrieve) | format_docs, 'question': RunnablePassthrough()}
    | RAG_PROMPT | llm | StrOutputParser()
)

CHAINS = {
    'Baseline (Bi-encoder)': (chain_baseline, baseline_retriever.invoke),
    'Reranker (Cross-encoder)': (chain_reranked, reranker.retrieve),
    'HyDE': (chain_hyde, hyde_retriever.retrieve),
}

print('Done!')

loading model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

Done!


## Gradio UI

Three panels:
- Top-left: Method selector + chat interface
- Right: Retrieved Context (updated with each response)
- Bottom: Method description cards

In [4]:
import gradio as gr

def respond(message, history, method):

    if not message.strip():
        return history, ""

    chain, retriever_fn = CHAINS[method]

    docs = retriever_fn(message)

    answer = chain.invoke(message)

    context_text = f"**Method: {method}**\n"
    context_text += f"Query: *{message}*\n\n"
    context_text += "---\n"
    for i, doc in enumerate(docs, 1):
        content = doc.page_content
        if 'Answer:' in content:
            content = content.split('Answer:')[1].strip()
        context_text += f"**[{i}]** {content[:200]}\n\n"

    history = history + [(message, answer)]
    return history, context_text


METHOD_DESCRIPTIONS = {
    'Baseline (Bi-encoder)': (
        "**Bi-encoder**\n\n"
        "Query and documents are embedded separately; cosine similarity is computed between them.\n"
        "Fastest approach; works best when query phrasing closely matches document wording."
    ),
    'Reranker (Cross-encoder)': (
        "**Cross-encoder Reranker**\n\n"
        "The bi-encoder first retrieves the top-15 candidates; the cross-encoder then scores each\n"
        "(query, doc) pair and re-ranks them. Provides more accurate semantic understanding."
    ),
    'HyDE': (
        "**HyDE**\n\n"
        "An LLM first generates a hypothetical answer; its embedding is then used to search real documents.\n"
        "Works best for colloquial or vague queries."
    ),
}

def update_method_desc(method):
    return METHOD_DESCRIPTIONS[method]


# Gradio UI
with gr.Blocks(title="Shopee RAG Customer Service") as demo:

    gr.Markdown("""
    # Shopee Customer Service RAG
    Compare three retrieval methods: Baseline / Reranker / HyDE
""")

    with gr.Row():

        # Left panel: chat
        with gr.Column(scale=3):

            method_selector = gr.Radio(
                choices=list(CHAINS.keys()),
                value='Baseline (Bi-encoder)',
                label='Retrieval Method',
                info='Switching methods takes effect on the next message'
            )

            method_desc = gr.Markdown(
                value=METHOD_DESCRIPTIONS['Baseline (Bi-encoder)']
            )

            chatbot = gr.Chatbot(
                label='Chat',
                height=420,
                show_label=False,
            )

            with gr.Row():
                msg_input = gr.Textbox(
                    placeholder='Ask about orders, returns, payments...',
                    label='',
                    scale=5,
                    lines=1,
                )
                send_btn = gr.Button('Send', scale=1, variant='primary')
                clear_btn = gr.Button('Clear', scale=1)

            gr.Examples(
                examples=[
                    'How do I return an item?',
                    'I got the wrong thing, can I send it back?',
                    'my parcel is taking forever',
                    'seller went MIA and I want refund',
                    'box was smashed when it arrived',
                ],
                inputs=msg_input,
                label='Try these examples'
            )

        # Right panel: Retrieved Context
        with gr.Column(scale=2):
            context_panel = gr.Markdown(
                value="*Retrieved context will appear here after your first message.*",
                label='Retrieved Context',
            )
            gr.Markdown("""
            ---
            **About this panel**

            Shows the top-3 documents retrieved from the knowledge base.
            The LLM answers based **only** on these documents.
            """)


    method_selector.change(
        fn=update_method_desc,
        inputs=method_selector,
        outputs=method_desc,
    )

    send_inputs  = [msg_input, chatbot, method_selector]
    send_outputs = [chatbot, context_panel]

    msg_input.submit(
        fn=respond,
        inputs=send_inputs,
        outputs=send_outputs,
    ).then(
        fn=lambda: '',
        outputs=msg_input,
    )

    send_btn.click(
        fn=respond,
        inputs=send_inputs,
        outputs=send_outputs,
    ).then(
        fn=lambda: '',
        outputs=msg_input,
    )

    clear_btn.click(
        fn=lambda: ([], '*Retrieved context will appear here after your first message.*'),
        outputs=[chatbot, context_panel],
    )


demo.launch(share=True, debug=False)

/tmp/ipykernel_4857/365699124.py:73: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot(
/tmp/ipykernel_4857/365699124.py:73: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to disable tags in your chatbot.
  chatbot = gr.Chatbot(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://346dc04c8238cd2a3a.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
